# Squash Video Classification: ResNet-18 + GRU

This notebook trains a **video-level, three-class classifier** for:

- `no_let`
- `stroke`
- `yes_let`

Each video is treated as one training example. The notebook samples frames in temporal order, extracts frozen features with an ImageNet-pretrained ResNet, and trains a GRU to classify the complete sequence.

**Important:** the 80/20 split is performed on complete video files *before* frame extraction. Frames from one video can therefore never appear in both train and test sets.

Run the cells from top to bottom. By default, the notebook expects this structure relative to the notebook's working directory:

```text
AI project/
├── squash_video_classification.ipynb
├── dataset/
│   ├── no_let/
│   ├── stroke/
│   └── yes_let/
└── runs/                 # created automatically
```

The file scanner accepts `.mov` and `.mp4` (as well as `.m4v`, `.avi`, and `.mkv`) so clips are not accidentally ignored if their extensions differ.

## 1. Kaggle environment check

Kaggle already provides CUDA-enabled PyTorch and the common scientific Python packages. Do not reinstall `torch` or `torchvision` here, because doing so can replace the compatible CUDA build. The next code cell only checks the installed versions.


In [1]:
import torch
import torchvision
import cv2
import sklearn

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("OpenCV:", cv2.__version__)
print("scikit-learn:", sklearn.__version__)


PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
OpenCV: 4.14.0
scikit-learn: 1.6.1


## 2. Imports and configuration

The dataset is attached as a Kaggle input. Its path is read-only. All generated files are saved under `/kaggle/working`, which is the writable output area.


In [2]:
from pathlib import Path
import os
import csv
import json
import math
import hashlib
import random
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Kaggle input path copied from the attached dataset shown in your screenshot.
PROJECT_DIR = Path("/kaggle/working")
DATA_DIR = Path("/kaggle/input/datasets/a5traeus/squash-fouls/dataset")
OUTPUT_DIR = Path("/kaggle/working/runs/squash_gru")

# Training configuration
TEST_SIZE = 0.20
NUM_FRAMES = 24
BACKBONE_NAME = "resnet50"     # "resnet18" or "resnet50"
GRU_HIDDEN_SIZE = 128
DROPOUT = 0.35
EPOCHS = 50
BATCH_SIZE = 8                 # batch of videos for GRU training
FEATURE_BATCH_SIZE = 16        # batch of frames through the CNN; safe starting point for a Kaggle GPU
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
RANDOM_SEED = 42
NUM_WORKERS = 0                # feature tensors are already cached in memory; 0 is reliable on Kaggle

# Support both extensions seen in the supplied file listing, plus common variants.
VIDEO_EXTENSIONS = {".mov", ".mp4", ".m4v", ".avi", ".mkv"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

Project directory: /kaggle/working
Dataset directory: /kaggle/input/datasets/a5traeus/squash-fouls/dataset
Output directory: /kaggle/working/runs/squash_gru


## 3. Select the Kaggle GPU and set random seeds

The notebook selects CUDA automatically. This cell also prints the GPU model and stops with a clear message if the Kaggle session is still running on CPU.


In [3]:
def set_seed(seed: int = RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        return torch.device("xpu")
    return torch.device("cpu")


set_seed()
torch.set_num_threads(max(1, min(8, os.cpu_count() or 4)))
DEVICE = choose_device()
print("PyTorch version:", torch.__version__)
print("Selected device:", DEVICE)
if DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA GPU not detected. In Kaggle, open Notebook > Session options, "
        "set Accelerator to a GPU, wait for the session to restart, then rerun this cell."
    )
print("GPU name:", torch.cuda.get_device_name(0))
print("CUDA version used by PyTorch:", torch.version.cuda)
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

PyTorch version: 2.11.0+cu130
Selected device: cuda
GPU name: Tesla T4
CUDA version used by PyTorch: 13.0
GPU memory (GB): 14.56


## 4. Discover videos and inspect class balance

Every immediate subfolder inside `dataset` is treated as a class. The folder name becomes the label. The scanner reports counts by class and extension so missing or unsupported files are visible before training.

In [4]:
def discover_videos(data_dir: Path):
    if not data_dir.exists() or not data_dir.is_dir():
        raise FileNotFoundError(
            f"Dataset folder not found: {data_dir}\n"
            "Expected dataset/<class_name>/<video files>. "
            "Update DATA_DIR in the configuration cell if needed."
        )

    class_folders = sorted(
        folder for folder in data_dir.iterdir()
        if folder.is_dir() and not folder.name.startswith(".")
    )

    class_names = []
    video_paths = []
    labels = []
    summary_rows = []

    for folder in class_folders:
        clips = sorted(
            path for path in folder.iterdir()
            if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
        )
        if not clips:
            print(f"WARNING: no supported video files found in {folder.name!r}")
            continue

        class_index = len(class_names)
        class_names.append(folder.name)
        video_paths.extend(clips)
        labels.extend([class_index] * len(clips))

        extension_counts = Counter(path.suffix.lower() for path in clips)
        summary_rows.append({
            "class": folder.name,
            "videos": len(clips),
            "extensions": dict(extension_counts),
        })

    if len(class_names) < 2:
        raise RuntimeError(
            f"Found only {len(class_names)} class folder(s) containing supported videos. "
            f"Supported extensions: {sorted(VIDEO_EXTENSIONS)}"
        )

    return video_paths, class_names, labels, summary_rows


video_paths, class_names, labels, class_summary = discover_videos(DATA_DIR)
summary_df = pd.DataFrame(class_summary)
display(summary_df)

print(f"Total videos: {len(video_paths)}")
print("Class-to-index mapping:", {name: i for i, name in enumerate(class_names)})

,class,videos,extensions
0,no_let,80,{'.mov': 80}
1,stroke,104,"{'.mp4': 88, '.mov': 16}"
2,yes_let,88,{'.mov': 88}


Total videos: 272
Class-to-index mapping: {'no_let': 0, 'stroke': 1, 'yes_let': 2}


## 5. Inspect sample video metadata

This checks whether OpenCV can open at least one file from each class and reports its resolution, frame rate, frame count, and approximate duration. If a video cannot be opened, verify that its codec is supported by your OpenCV/FFmpeg installation.

In [5]:
def inspect_video(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {
            "video": str(video_path), "opened": False,
            "width": None, "height": None, "fps": None,
            "frame_count": None, "duration_seconds": None,
        }

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    duration = frame_count / fps if fps > 0 and frame_count > 0 else None
    return {
        "video": str(video_path), "opened": True,
        "width": width, "height": height, "fps": fps,
        "frame_count": frame_count, "duration_seconds": duration,
    }


sample_rows = []
for class_name in class_names:
    class_files = [p for p, y in zip(video_paths, labels) if class_names[y] == class_name]
    sample_rows.append(inspect_video(class_files[0]))

metadata_df = pd.DataFrame(sample_rows)
display(metadata_df)
if not metadata_df["opened"].all():
    print("WARNING: At least one sample video could not be opened. Resolve codec/file issues before continuing.")

,video,opened,width,height,fps,frame_count,duration_seconds
0,/kaggle/input/datasets/a5traeus/squash-fouls/d...,True,1920,1080,30.0,166,5.533333
1,/kaggle/input/datasets/a5traeus/squash-fouls/d...,True,1920,1080,60.0,252,4.200000
2,/kaggle/input/datasets/a5traeus/squash-fouls/d...,True,1080,1350,30.0,145,4.833333


## 6. Make the stratified 80/20 split at video level

The split is done using video paths and their folder labels. Unequal class sizes are supported. With stratification, each class is represented in both splits, with exact percentages subject to integer rounding.

In [6]:
if not 0.05 <= TEST_SIZE <= 0.50:
    raise ValueError("TEST_SIZE should be between 0.05 and 0.50.")

label_array = np.asarray(labels, dtype=np.int64)
unique_labels, class_counts = np.unique(label_array, return_counts=True)
if np.any(class_counts < 2):
    too_small = {
        class_names[int(label)]: int(count)
        for label, count in zip(unique_labels, class_counts) if count < 2
    }
    raise ValueError(f"Every class needs at least two videos for stratified splitting: {too_small}")

n_test = math.ceil(len(video_paths) * TEST_SIZE)
n_train = len(video_paths) - n_test
if n_test < len(class_names) or n_train < len(class_names):
    raise ValueError("Not enough total videos to place every class in both train and test splits.")

train_paths, test_paths, train_labels, test_labels = train_test_split(
    video_paths,
    label_array.tolist(),
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=label_array,
)

split_rows = []
for split_name, paths, split_labels in [
    ("train", train_paths, train_labels),
    ("test", test_paths, test_labels),
]:
    for path, label in zip(paths, split_labels):
        split_rows.append({
            "split": split_name,
            "video_path": str(path.relative_to(DATA_DIR)),
            "class_name": class_names[int(label)],
            "label": int(label),
        })

split_df = pd.DataFrame(split_rows)
split_df.to_csv(OUTPUT_DIR / "data_split.csv", index=False)

split_counts = pd.crosstab(split_df["class_name"], split_df["split"])
for column in ["train", "test"]:
    if column not in split_counts.columns:
        split_counts[column] = 0
split_counts = split_counts[["train", "test"]]
split_counts["total"] = split_counts.sum(axis=1)
display(split_counts)
print(f"Train videos: {len(train_paths)} | Test videos: {len(test_paths)}")
print("Saved split manifest:", OUTPUT_DIR / "data_split.csv")

split,train,test,total
class_name,,,
no_let,64,16,80
stroke,83,21,104
yes_let,70,18,88


Train videos: 217 | Test videos: 55
Saved split manifest: /kaggle/working/runs/squash_gru/data_split.csv
